# Figure 1: Critical coupling from the susceptibility

This notebook tests Eq. (24) of the manuscript for the Hopf-rotation
ensemble on hyperspheres. We evaluate
\(s_{S^D}(\lambda;g_\Delta)\) and compare \(1/s_{S^D}\) with
\(K_c=1/[\kappa(S^{D})\pi g_\Delta(0)]\). Here \(D=\dim M\) follows
the notation of the manuscript. The Hopf-rotation ensemble is
implemented in the even ambient dimensions \(D_a=D+1\). This directly tests the
calculation of \(K_c\) used in the manuscript.

This figure is a spectral-response test and has no finite-\(N\)
system-size parameter. Finite-size effects are tested separately in
Figure 2.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.lines import Line2D

# Set the experiment root for runs started from the notebook directory or the
# experiment directory.
HERE = Path.cwd().resolve()
EXPERIMENT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
SRC_DIR = EXPERIMENT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import DATA_DIR, FIGURE_DIR, ensure_directories
from plotting import set_paper_style, save_figure
from manifold_sync import *

ensure_directories()
set_paper_style()


def manuscript_palette(labels):
    # Return the blue-green manuscript palette used for dimension-like groups.
    labels = list(labels)
    colors = sns.color_palette("viridis", n_colors=len(labels))
    return dict(zip(labels, colors))

In [ ]:
# The shared Hopf helpers use the ambient dimension n for S^{n-1}.
# The manuscript uses D = dim(M), so the plotted dimensions are n - 1.
AMBIENT_DIMENSIONS = [2, 4, 6, 8, 10]
DELTAS = np.linspace(0.03, 1.20, 500)
DISTRIBUTIONS = ["lorentzian", "gaussian", "uniform"]
EPSILON_FACTORS = [
    0.30, 0.24, 0.18, 0.135, 0.10, 0.075,
    0.055, 0.040, 0.030, 0.022, 0.016, 0.012,
    0.009, 0.006, 0.004, 0.0025, 0.0015, 0.001,
]

print(
    f"Planned susceptibility data points: "
    f"{len(AMBIENT_DIMENSIONS) * len(DELTAS) * len(DISTRIBUTIONS) * len(EPSILON_FACTORS):,}"
)

In [ ]:
spectral = scan_spectral_thresholds(
    D_values=AMBIENT_DIMENSIONS,
    delta_values=DELTAS,
    distributions=DISTRIBUTIONS,
    epsilon_factors=EPSILON_FACTORS,
)
spectral = spectral.rename(columns={"D": "ambient_dimension"})
spectral["manifold_dimension"] = spectral["ambient_dimension"] - 1

min_epsilon = float(np.min(EPSILON_FACTORS))
summary = spectral.loc[
    np.isclose(spectral["epsilon_factor"], min_epsilon)
].copy()

raw_path = DATA_DIR / "fig01_spectral_raw.csv"
summary_path = DATA_DIR / "fig01_threshold_summary.csv"
spectral.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)

density_labels = {
    "lorentzian": "Lorentzian",
    "gaussian": "Gaussian",
    "uniform": "Uniform",
}
spectral["Probability density"] = spectral["distribution"].map(density_labels)
summary["Probability density"] = summary["distribution"].map(density_labels)
spectral["Manifold dimension"] = "D = " + spectral["manifold_dimension"].astype(str)
summary["Manifold dimension"] = "D = " + summary["manifold_dimension"].astype(str)

summary.head()

In [ ]:
TITLE_SIZE = 9.5
LABEL_SIZE = 9.5
TICK_SIZE = 8.5
LEGEND_SIZE = 7.5
LEGEND_TITLE_SIZE = 8.5
PANEL_LABEL_SIZE = 12

fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.25))

density_order = ["Lorentzian", "Gaussian", "Uniform"]
palette = dict(zip(density_order, sns.color_palette("Set2", n_colors=3)))
D_order = sorted(spectral["manifold_dimension"].unique())
dimension_palette = dict(zip(D_order, sns.color_palette("viridis", n_colors=len(D_order))))
dimension_label_palette = {
    f"D = {D}": dimension_palette[D]
    for D in D_order
}
markers = ["o", "s", "^", "D", "P", "X", "v", "<", ">"]
marker_map = {D: markers[i % len(markers)] for i, D in enumerate(D_order)}
legend_style = dict(
    frameon=True,
    fontsize=LEGEND_SIZE,
    title_fontsize=LEGEND_TITLE_SIZE,
    borderpad=0.35,
    labelspacing=0.25,
    handlelength=1.4,
    handletextpad=0.45,
)

convergence = spectral.groupby(
    ["Probability density", "epsilon_factor"], as_index=False
)["collapse"].mean()
sns.lineplot(
    data=convergence,
    x="epsilon_factor",
    y="collapse",
    hue="Probability density",
    hue_order=density_order,
    palette=palette,
    errorbar=None,
    lw=1.4,
    ax=axes[0],
    legend=False,
)

convergence_points = spectral.groupby(
    ["Probability density", "manifold_dimension", "epsilon_factor"], as_index=False
)["collapse"].mean()
center = (len(D_order) - 1) / 2.0
offsets = {D: np.exp((i - center) * 0.045) for i, D in enumerate(D_order)}
convergence_points["epsilon_plot"] = convergence_points.apply(
    lambda row: row["epsilon_factor"] * offsets[row["manifold_dimension"]],
    axis=1,
)
for D in D_order:
    for density in density_order:
        part = convergence_points[
            (convergence_points["manifold_dimension"] == D)
            & (convergence_points["Probability density"] == density)
        ]
        axes[0].scatter(
            part["epsilon_plot"],
            part["collapse"],
            s=13,
            marker=marker_map[D],
            color=palette[density],
            edgecolor="none",
            alpha=0.72,
        )
axes[0].set_xscale("log")
axes[0].invert_xaxis()
major_tick_candidates = [0.30, 0.10, 0.03, 0.01, 0.003, 0.001]
major_ticks = [
    tick
    for tick in major_tick_candidates
    if min(EPSILON_FACTORS) <= tick <= max(EPSILON_FACTORS)
]
axes[0].xaxis.set_major_locator(mticker.FixedLocator(major_ticks))
axes[0].xaxis.set_minor_formatter(mticker.NullFormatter())
axes[0].set_xticklabels([f"{tick:g}" for tick in major_ticks])
axes[0].set_xlabel(r"$\lambda/\Delta$")
axes[0].set_ylabel(r"$K_c(\lambda)/K_c(0^+)$")
axes[0].set_title(r"Limit as $\lambda\to0^+$")

density_handles = [
    Line2D([0], [0], color=palette[label], lw=1.6, label=label)
    for label in density_order
]
dimension_handles = [
    Line2D(
        [0],
        [0],
        marker=marker_map[D],
        color="black",
        linestyle="None",
        markersize=4,
        label=f"D = {D}",
    )
    for D in D_order
]
leg0 = axes[0].legend(
    handles=density_handles,
    title="Probability density",
    loc="upper right",
    **legend_style,
)
axes[0].add_artist(leg0)
axes[0].legend(
    handles=dimension_handles,
    title="Manifold dimension",
    loc="lower right",
    **legend_style,
)

lorentzian = summary[summary["distribution"] == "lorentzian"].copy()
sns.lineplot(
    data=lorentzian,
    x="delta",
    y="Kc_theory",
    hue="Manifold dimension",
    palette=dimension_label_palette,
    ax=axes[1],
    legend=False,
    lw=1.35,
    ls="--",
    zorder=5,
)
marker_indices = np.unique(
    np.linspace(0, lorentzian["delta"].nunique() - 1, 12, dtype=int)
)
marker_deltas = np.sort(lorentzian["delta"].unique())[marker_indices]
lorentzian_markers = lorentzian[lorentzian["delta"].isin(marker_deltas)]
for D in D_order:
    part = lorentzian_markers[lorentzian_markers["manifold_dimension"] == D]
    axes[1].scatter(
        part["delta"],
        part["Kc_numeric"],
        s=18,
        facecolors="white",
        edgecolors=dimension_palette[D],
        linewidth=0.75,
        zorder=4,
    )
axes[1].set_xlabel(r"$\Delta$")
axes[1].set_ylabel(r"$K_c$")
axes[1].set_title(r"$K_c$ for Lorentzian density")

D_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color=dimension_palette[D],
        linestyle="None",
        markersize=4,
        label=f"D = {D}",
    )
    for i, D in enumerate(D_order)
]
type_handles = [
    Line2D([0], [0], color="black", lw=1.2, ls="--", label="Theory"),
    Line2D(
        [0],
        [0],
        marker="o",
        markerfacecolor="white",
        markeredgecolor="black",
        color="black",
        linestyle="None",
        markersize=4,
        label="Numerical",
    ),
]
leg1 = axes[1].legend(
    handles=D_handles,
    title="Manifold dimension",
    loc="upper left",
    **legend_style,
)
axes[1].add_artist(leg1)
axes[1].legend(
    handles=type_handles,
    title="Data type",
    loc="lower right",
    **legend_style,
)

collapse_by_dimension = summary.groupby(
    ["Probability density", "manifold_dimension", "Manifold dimension"], as_index=False
)["collapse"].mean()
x_positions = {label: i for i, label in enumerate(density_order)}
x_offsets = {
    D: (i - (len(D_order) - 1) / 2.0) * 0.08
    for i, D in enumerate(D_order)
}
for D in D_order:
    part = collapse_by_dimension[collapse_by_dimension["manifold_dimension"] == D]
    axes[2].scatter(
        [x_positions[label] + x_offsets[D] for label in part["Probability density"]],
        part["collapse"],
        s=24,
        color=dimension_palette[D],
        edgecolor="white",
        linewidth=0.35,
        label=f"D = {D}",
        zorder=4,
    )
axes[2].set_xticks(range(len(density_order)))
axes[2].set_xticklabels(density_order)
axes[2].set_xlabel("Probability density")
axes[2].set_ylabel(r"$K_c^{\rm num}/K_c^{\rm th}$")
axes[2].set_title(r"Normalized $K_c$")
axes[2].legend(
    title="Manifold dimension",
    loc="lower center",
    ncol=3,
    columnspacing=0.8,
    **legend_style,
)
collapse_low = summary["collapse"].min()
collapse_high = summary["collapse"].max()
collapse_pad = max(2.0e-4, 0.4 * (collapse_high - collapse_low))
axes[2].set_ylim(collapse_low - collapse_pad, collapse_high + collapse_pad)
axes[2].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))

for ax in axes:
    ax.title.set_fontsize(TITLE_SIZE)
    ax.xaxis.label.set_size(LABEL_SIZE)
    ax.yaxis.label.set_size(LABEL_SIZE)
    ax.tick_params(axis="both", labelsize=TICK_SIZE)
    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontsize(TICK_SIZE)

fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.92], w_pad=1.4)
for label, ax in zip(["(a)", "(b)", "(c)"], axes):
    box = ax.get_position()
    fig.text(
        box.x0,
        box.y1 + 0.020,
        label,
        ha="left",
        va="bottom",
        fontsize=PANEL_LABEL_SIZE,
        fontweight="normal",
    )
save_figure(fig, FIGURE_DIR / "fig01_linear_threshold")
fig.savefig(FIGURE_DIR / "fig01_linear_threshold_600dpi.png", dpi=600, bbox_inches="tight")